Step 6:

In [2]:
import os
import numpy as np
import pandas as pd


os.makedirs("results", exist_ok=True)

cp_path = "/content/clusterProfiler_GO_BP_ANCG_SZBPD_vs_control.tsv"
topgo_path = "/content/step5_topGo_BP.csv"
gprof_up_path = "/content/step5_GProfiler_up.csv"
gprof_down_path = "/content/step5_GProfiler_down.csv"

cp = pd.read_csv(cp_path, sep="\t")
topgo = pd.read_csv(topgo_path)
gprof_up = pd.read_csv(gprof_up_path)
gprof_down = pd.read_csv(gprof_down_path)

print("Loaded rows:")
print("clusterProfiler:", len(cp))
print("topGO:", len(topgo))
print("gProfiler2 up:", len(gprof_up))
print("gProfiler2 down:", len(gprof_down))

print("\nColumn names:")
print("clusterProfiler:", cp.columns.tolist())
print("topGO:", topgo.columns.tolist())
print("gProfiler2 up:", gprof_up.columns.tolist())
print("gProfiler2 down:", gprof_down.columns.tolist())

cp_small = cp[
    [
        "ID",
        "Description",
        "GeneRatio",
        "BgRatio",
        "FoldEnrichment",
        "Count",
        "pvalue",
        "p.adjust"
    ]
].copy()

cp_small = cp_small.rename(columns={
    "ID": "GO_ID",
    "Description": "Term",
    "GeneRatio": "clusterProfiler_GeneRatio",
    "BgRatio": "clusterProfiler_BgRatio",
    "FoldEnrichment": "clusterProfiler_FoldEnrichment",
    "Count": "clusterProfiler_DE_gene_count",
    "pvalue": "clusterProfiler_raw_pvalue",
    "p.adjust": "clusterProfiler_BH_FDR"
})

cp_small["Ontology"] = "GO:BP"
cp_small["clusterProfiler_included"] = True
cp_small["clusterProfiler_significant"] = (
    cp_small["clusterProfiler_BH_FDR"] < 0.05
)


topgo.columns = topgo.columns.astype(str).str.strip()

topgo_id_col = next(
    (
        col for col in topgo.columns
        if col.lower().replace(".", "").replace("_", "") in {"goid", "goidentifier"}
    ),
    None
)

topgo_term_col = next(
    (
        col for col in topgo.columns
        if col.lower() in {"term", "description"}
    ),
    None
)

topgo_p_col = next(
    (
        col for col in topgo.columns
        if col.lower().replace(".", "").replace("_", "") in {"pvalue", "pval"}
    ),
    None
)

if topgo_id_col is None or topgo_term_col is None or topgo_p_col is None:
    raise ValueError(
        "Could not identify topGO GO-ID, Term, and p-value columns.\n"
        f"Found columns: {topgo.columns.tolist()}"
    )

topgo_keep = [topgo_id_col, topgo_term_col, topgo_p_col]

for col in ["Annotated", "Significant", "Expected", "padj_BH", "p.adjust"]:
    if col in topgo.columns:
        topgo_keep.append(col)

topgo_small = topgo[topgo_keep].copy().rename(columns={
    topgo_id_col: "GO_ID",
    topgo_term_col: "topGO_Term",
    topgo_p_col: "topGO_raw_pvalue",
    "Annotated": "topGO_Annotated",
    "Significant": "topGO_DE_gene_count",
    "Expected": "topGO_Expected_DE_genes",
    "padj_BH": "topGO_BH_FDR",
    "p.adjust": "topGO_BH_FDR"
})

topgo_small["topGO_raw_pvalue"] = pd.to_numeric(
    topgo_small["topGO_raw_pvalue"]
        .astype(str)
        .str.replace("<", "", regex=False)
        .str.strip(),
    errors="coerce"
)

if "topGO_BH_FDR" in topgo_small.columns:
    topgo_small["topGO_BH_FDR"] = pd.to_numeric(
        topgo_small["topGO_BH_FDR"],
        errors="coerce"
    )
    topgo_small["topGO_significant"] = topgo_small["topGO_BH_FDR"] < 0.05
    topgo_significance_rule = "topGO BH FDR < 0.05"
else:
    topgo_small["topGO_significant"] = topgo_small["topGO_raw_pvalue"] < 0.01
    topgo_significance_rule = "topGO raw p-value < 0.01"

topgo_small["topGO_included"] = True
topgo_small["Ontology_topGO"] = "GO:BP"

topgo_small = (
    topgo_small
    .sort_values("topGO_raw_pvalue")
    .drop_duplicates(subset="GO_ID", keep="first")
)


def prepare_gprofiler(df, direction):
    df = df.copy()
    df.columns = df.columns.astype(str).str.strip()

    required = ["native", "name", "p_value"]
    missing = [col for col in required if col not in df.columns]

    if missing:
        raise ValueError(
            f"gProfiler2 {direction} file is missing: {missing}\n"
            f"Found columns: {df.columns.tolist()}"
        )

    keep_cols = ["native", "name", "p_value"]

    for col in [
        "source",
        "significant",
        "intersection_size",
        "query_size",
        "term_size",
        "effective_domain_size",
        "precision",
        "recall"
    ]:
        if col in df.columns:
            keep_cols.append(col)

    out = df[keep_cols].copy().rename(columns={
        "native": "GO_ID",
        "name": f"gProfiler_{direction}_Term",
        "p_value": f"gProfiler_{direction}_FDR",
        "source": f"gProfiler_{direction}_Ontology",
        "significant": f"gProfiler_{direction}_significant_flag",
        "intersection_size": f"gProfiler_{direction}_intersection_size",
        "query_size": f"gProfiler_{direction}_query_size",
        "term_size": f"gProfiler_{direction}_term_size",
        "effective_domain_size": f"gProfiler_{direction}_background_size",
        "precision": f"gProfiler_{direction}_precision",
        "recall": f"gProfiler_{direction}_recall"
    })

    out[f"gProfiler_{direction}_FDR"] = pd.to_numeric(
        out[f"gProfiler_{direction}_FDR"],
        errors="coerce"
    )

    out[f"gProfiler_{direction}_included"] = True


    flag_col = f"gProfiler_{direction}_significant_flag"

    if flag_col in out.columns:
        out[f"gProfiler_{direction}_significant"] = (
            out[flag_col]
            .astype(str)
            .str.strip()
            .str.lower()
            .eq("true")
        )
    else:
        out[f"gProfiler_{direction}_significant"] = (
            out[f"gProfiler_{direction}_FDR"] < 0.05
        )

    out = (
        out
        .sort_values(f"gProfiler_{direction}_FDR")
        .drop_duplicates(subset="GO_ID", keep="first")
    )

    return out


gprof_up_small = prepare_gprofiler(gprof_up, "Up")
gprof_down_small = prepare_gprofiler(gprof_down, "Down")

gprof_both = gprof_up_small.merge(
    gprof_down_small,
    on="GO_ID",
    how="outer"
)

gprof_term_cols = [
    col for col in ["gProfiler_Up_Term", "gProfiler_Down_Term"]
    if col in gprof_both.columns
]
gprof_both["gProfiler_Term"] = gprof_both[gprof_term_cols].bfill(axis=1).iloc[:, 0]

gprof_ontology_cols = [
    col for col in ["gProfiler_Up_Ontology", "gProfiler_Down_Ontology"]
    if col in gprof_both.columns
]

if gprof_ontology_cols:
    gprof_both["gProfiler_Ontology"] = (
        gprof_both[gprof_ontology_cols].bfill(axis=1).iloc[:, 0]
    )
else:
    gprof_both["gProfiler_Ontology"] = pd.NA

up_included = gprof_both.get(
    "gProfiler_Up_included",
    pd.Series(False, index=gprof_both.index)
).fillna(False).astype(bool)

down_included = gprof_both.get(
    "gProfiler_Down_included",
    pd.Series(False, index=gprof_both.index)
).fillna(False).astype(bool)

up_significant = gprof_both.get(
    "gProfiler_Up_significant",
    pd.Series(False, index=gprof_both.index)
).fillna(False).astype(bool)

down_significant = gprof_both.get(
    "gProfiler_Down_significant",
    pd.Series(False, index=gprof_both.index)
).fillna(False).astype(bool)

gprof_both["gProfiler_included"] = up_included | down_included
gprof_both["gProfiler_significant"] = up_significant | down_significant


joint = cp_small.merge(
    topgo_small,
    on="GO_ID",
    how="outer"
)

joint["Term"] = joint[["Term", "topGO_Term"]].bfill(axis=1).iloc[:, 0]

joint = joint.merge(
    gprof_both,
    on="GO_ID",
    how="outer"
)

joint["Term"] = joint[["Term", "gProfiler_Term"]].bfill(axis=1).iloc[:, 0]

joint["Ontology"] = joint[
    ["Ontology", "Ontology_topGO", "gProfiler_Ontology"]
].bfill(axis=1).iloc[:, 0]


boolean_cols = [
    "clusterProfiler_included",
    "topGO_included",
    "gProfiler_included",
    "clusterProfiler_significant",
    "topGO_significant",
    "gProfiler_significant"
]

for col in boolean_cols:
    if col not in joint.columns:
        joint[col] = False
    joint[col] = joint[col].fillna(False).astype(bool)

joint["Methods included"] = (
    joint["clusterProfiler_included"].astype(int)
    + joint["topGO_included"].astype(int)
    + joint["gProfiler_included"].astype(int)
)

joint["Methods significantly enriched"] = (
    joint["clusterProfiler_significant"].astype(int)
    + joint["topGO_significant"].astype(int)
    + joint["gProfiler_significant"].astype(int)
)

def get_significant_methods(row):
    methods = []

    if row["clusterProfiler_significant"]:
        methods.append("clusterProfiler")

    if row["topGO_significant"]:
        methods.append("topGO")

    if row["gProfiler_significant"]:
        directions = []

        if row.get("gProfiler_Up_significant", False):
            directions.append("Up")

        if row.get("gProfiler_Down_significant", False):
            directions.append("Down")

        direction_label = "/".join(directions)
        methods.append(f"gProfiler2 ({direction_label})")

    return "; ".join(methods)

joint["Significant method(s)"] = joint.apply(
    get_significant_methods,
    axis=1
)


preferred_columns = [
    "GO_ID",
    "Term",
    "Ontology",

    "clusterProfiler_BH_FDR",
    "clusterProfiler_raw_pvalue",
    "clusterProfiler_FoldEnrichment",
    "clusterProfiler_DE_gene_count",

    "topGO_raw_pvalue",
    "topGO_BH_FDR",
    "topGO_Annotated",
    "topGO_DE_gene_count",
    "topGO_Expected_DE_genes",

    "gProfiler_Up_FDR",
    "gProfiler_Up_intersection_size",
    "gProfiler_Down_FDR",
    "gProfiler_Down_intersection_size",

    "Methods significantly enriched",
    "Methods included",
    "Significant method(s)",

    "clusterProfiler_included",
    "topGO_included",
    "gProfiler_included",
    "clusterProfiler_significant",
    "topGO_significant",
    "gProfiler_significant"
]

final_columns = [col for col in preferred_columns if col in joint.columns]

joint["_best_pvalue"] = joint[
    [
        col for col in [
            "clusterProfiler_BH_FDR",
            "topGO_raw_pvalue",
            "gProfiler_Up_FDR",
            "gProfiler_Down_FDR"
        ]
        if col in joint.columns
    ]
].min(axis=1, skipna=True)

joint = (
    joint
    .sort_values(
        by=[
            "Methods significantly enriched",
            "Methods included",
            "_best_pvalue"
        ],
        ascending=[False, False, True],
        na_position="last"
    )
    .drop(columns="_best_pvalue")
    .loc[:, final_columns]
    .reset_index(drop=True)
)

output_path = "results/step6_joint_GO_enrichment_all_methods.tsv"

joint.to_csv(
    output_path,
    sep="\t",
    index=False
)


print("\n--- Step 6 completed ---")
print("topGO significance rule:", topgo_significance_rule)
print("Unique GO terms across all methods:", len(joint))
print(
    "Terms significantly enriched by >= 2 methods:",
    (joint["Methods significantly enriched"] >= 2).sum()
)
print("Saved full Step 6 table to:", output_path)

display(joint.head(10))

Loaded rows:
clusterProfiler: 82
topGO: 6131
gProfiler2 up: 11
gProfiler2 down: 194

Column names:
clusterProfiler: ['ID', 'Description', 'GeneRatio', 'BgRatio', 'RichFactor', 'FoldEnrichment', 'zScore', 'pvalue', 'p.adjust', 'qvalue', 'geneID', 'Count']
topGO: ['GO.ID', 'Term', 'Annotated', 'Significant', 'Expected', 'pvalue']
gProfiler2 up: ['native', 'name', 'p_value', 'intersection_size', 'query_size', 'term_size', 'intersections', 'evidences']
gProfiler2 down: ['native', 'name', 'p_value', 'intersection_size', 'query_size', 'term_size', 'intersections', 'evidences']


/tmp/ipykernel_1151/3886748214.py:243: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  ).fillna(False).astype(bool)
/tmp/ipykernel_1151/3886748214.py:248: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  ).fillna(False).astype(bool)
/tmp/ipykernel_1151/3886748214.py:253: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  ).fillna(False).astype(bool


--- Step 6 completed ---
topGO significance rule: topGO raw p-value < 0.01
Unique GO terms across all methods: 6201
Terms significantly enriched by >= 2 methods: 54
Saved full Step 6 table to: results/step6_joint_GO_enrichment_all_methods.tsv


/tmp/ipykernel_1151/3886748214.py:297: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  joint[col] = joint[col].fillna(False).astype(bool)
/tmp/ipykernel_1151/3886748214.py:297: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  joint[col] = joint[col].fillna(False).astype(bool)
/tmp/ipykernel_1151/3886748214.py:297: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downca

,GO_ID,Term,Ontology,clusterProfiler_BH_FDR,clusterProfiler_raw_pvalue,clusterProfiler_FoldEnrichment,clusterProfiler_DE_gene_count,topGO_raw_pvalue,topGO_Annotated,topGO_DE_gene_count,...,gProfiler_Down_intersection_size,Methods significantly enriched,Methods included,Significant method(s),clusterProfiler_included,topGO_included,gProfiler_included,clusterProfiler_significant,topGO_significant,gProfiler_significant
0,GO:0071805,potassium ion transmembrane transport,GO:BP,0.001445,3.526792e-06,2.004476,45.0,4.100000e-04,199.0,45.0,...,40.0,3,3,clusterProfiler; topGO; gProfiler2 (Up/Down),True,True,True,True,True,True
1,GO:0050803,regulation of synapse structure or activity,GO:BP,0.004592,2.759261e-05,1.703550,59.0,7.000000e-04,309.0,59.0,...,51.0,3,3,clusterProfiler; topGO; gProfiler2 (Up/Down),True,True,True,True,True,True
2,GO:0007269,neurotransmitter secretion,GO:BP,0.001694,4.772754e-06,2.200342,35.0,8.010000e-03,141.0,35.0,...,32.0,3,3,clusterProfiler; topGO; gProfiler2 (Up/Down),True,True,True,True,True,True
3,GO:0007218,neuropeptide signaling pathway,GO:BP,0.000282,4.762855e-07,2.568143,31.0,4.500000e-07,107.0,31.0,...,27.0,3,3,clusterProfiler; topGO; gProfiler2 (Up/Down),True,True,True,True,True,True
4,GO:0007626,locomotory behavior,GO:BP,0.007825,6.318958e-05,2.407570,22.0,1.240000e-03,81.0,22.0,...,37.0,3,3,clusterProfiler; topGO; gProfiler2 (Up/Down),True,True,True,True,True,True
5,GO:0001508,action potential,GO:BP,0.003601,1.737787e-05,2.058790,36.0,7.900000e-05,156.0,36.0,...,31.0,3,3,clusterProfiler; topGO; gProfiler2 (Up/Down),True,True,True,True,True,True
6,GO:0050804,modulation of chemical synaptic transmission,GO:BP,0.000007,2.938652e-09,1.822199,96.0,1.515000e-02,468.0,96.0,...,90.0,2,3,clusterProfiler; gProfiler2 (Up/Down),True,True,True,True,False,True
7,GO:0099177,regulation of trans-synaptic signaling,GO:BP,0.000007,3.698425e-09,1.814428,96.0,1.000000e+00,470.0,96.0,...,90.0,2,3,clusterProfiler; gProfiler2 (Up/Down),True,True,True,True,False,True
8,GO:0061564,axon development,GO:BP,0.000020,1.884525e-08,1.838508,84.0,4.733900e-01,405.0,84.0,...,84.0,2,3,clusterProfiler; gProfiler2 (Up/Down),True,True,True,True,False,True
9,GO:0007610,behavior,GO:BP,0.002107,7.122202e-06,1.667756,73.0,1.000000e+00,391.0,73.0,...,96.0,2,3,clusterProfiler; gProfiler2 (Up/Down),True,True,True,True,False,True


Step 7:

In [4]:
import pandas as pd

step7_candidates = joint[
    joint["Methods significantly enriched"] >= 1
].copy()

pvalue_cols = [
    col for col in [
        "clusterProfiler_BH_FDR",
        "topGO_raw_pvalue",
        "gProfiler_Up_FDR",
        "gProfiler_Down_FDR"
    ]
    if col in step7_candidates.columns
]

step7_candidates["Best available p-value"] = (
    step7_candidates[pvalue_cols]
    .min(axis=1, skipna=True)
)

step7_candidates = step7_candidates.sort_values(
    by=[
        "Methods significantly enriched",
        "Methods included",
        "Best available p-value"
    ],
    ascending=[False, False, True],
    na_position="last"
)

step7_top10 = step7_candidates.head(10).copy()

step7_table = step7_top10[
    [
        "GO_ID",
        "Term",
        "Ontology",
        "clusterProfiler_BH_FDR",
        "clusterProfiler_FoldEnrichment",
        "topGO_raw_pvalue",
        "gProfiler_Up_FDR",
        "gProfiler_Down_FDR",
        "Methods significantly enriched",
        "Methods included",
        "Significant method(s)"
    ]
].copy()

step7_table = step7_table.rename(columns={
    "GO_ID": "GO ID",
    "Term": "GO Term",
    "Ontology": "Ontology",
    "clusterProfiler_BH_FDR": "clusterProfiler BH FDR",
    "clusterProfiler_FoldEnrichment": "clusterProfiler Fold Enrichment",
    "topGO_raw_pvalue": "topGO p-value",
    "gProfiler_Up_FDR": "gProfiler2 Up FDR",
    "gProfiler_Down_FDR": "gProfiler2 Down FDR",
    "Methods significantly enriched": "Methods Significant",
    "Methods included": "Methods Included",
    "Significant method(s)": "Significant Method(s)"
})

for col in [
    "clusterProfiler BH FDR",
    "topGO p-value",
    "gProfiler2 Up FDR",
    "gProfiler2 Down FDR"
]:
    if col in step7_table.columns:
        step7_table[col] = step7_table[col].apply(
            lambda x: f"{x:.2e}" if pd.notna(x) else ""
        )

if "clusterProfiler Fold Enrichment" in step7_table.columns:
    step7_table["clusterProfiler Fold Enrichment"] = (
        step7_table["clusterProfiler Fold Enrichment"]
        .apply(lambda x: f"{x:.2f}" if pd.notna(x) else "")
    )

step7_table = step7_table.reset_index(drop=True)

display(step7_table)

step7_table.to_csv(
    "results/step7_top10_GO_terms_most_methods.tsv",
    sep="\t",
    index=False
)

print(
    "Saved Step 7 table to: "
    "results/step7_top10_GO_terms_most_methods.tsv"
)

print("\nNumber of terms by number of significant methods:")
print(
    joint["Methods significantly enriched"]
    .value_counts()
    .sort_index(ascending=False)
)

,GO ID,GO Term,Ontology,clusterProfiler BH FDR,clusterProfiler Fold Enrichment,topGO p-value,gProfiler2 Up FDR,gProfiler2 Down FDR,Methods Significant,Methods Included,Significant Method(s)
0,GO:0071805,potassium ion transmembrane transport,GO:BP,1.44e-03,2.00,4.10e-04,,2.06e-08,3,3,clusterProfiler; topGO; gProfiler2 (Up/Down)
1,GO:0050803,regulation of synapse structure or activity,GO:BP,4.59e-03,1.70,7.00e-04,,4.43e-08,3,3,clusterProfiler; topGO; gProfiler2 (Up/Down)
2,GO:0007269,neurotransmitter secretion,GO:BP,1.69e-03,2.20,8.01e-03,,4.24e-07,3,3,clusterProfiler; topGO; gProfiler2 (Up/Down)
3,GO:0007218,neuropeptide signaling pathway,GO:BP,2.82e-04,2.57,4.50e-07,,4.71e-07,3,3,clusterProfiler; topGO; gProfiler2 (Up/Down)
4,GO:0007626,locomotory behavior,GO:BP,7.83e-03,2.41,1.24e-03,,1.35e-05,3,3,clusterProfiler; topGO; gProfiler2 (Up/Down)
5,GO:0001508,action potential,GO:BP,3.60e-03,2.06,7.90e-05,,5.21e-05,3,3,clusterProfiler; topGO; gProfiler2 (Up/Down)
6,GO:0050804,modulation of chemical synaptic transmission,GO:BP,6.56e-06,1.82,1.52e-02,,5.99e-18,2,3,clusterProfiler; gProfiler2 (Up/Down)
7,GO:0099177,regulation of trans-synaptic signaling,GO:BP,6.56e-06,1.81,1.00e+00,,6.88e-18,2,3,clusterProfiler; gProfiler2 (Up/Down)
8,GO:0061564,axon development,GO:BP,2.01e-05,1.84,4.73e-01,,3.94e-14,2,3,clusterProfiler; gProfiler2 (Up/Down)
9,GO:0007610,behavior,GO:BP,2.11e-03,1.67,1.00e+00,,9.52e-13,2,3,clusterProfiler; gProfiler2 (Up/Down)


Saved Step 7 table to: results/step7_top10_GO_terms_most_methods.tsv

Number of terms by number of significant methods:
Methods significantly enriched
3       6
2      48
1     244
0    5903
Name: count, dtype: int64
